In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import LabelEncoder

DATA_FOLDER = "/Users/shilppatel/Desktop/AA_FINAL_PROJECT"

# Load the combined dataset
df = pd.read_csv(os.path.join(DATA_FOLDER, "combined_dataset.csv"), low_memory=False)

print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")

Shape: 2,062,951 rows x 48 columns


In [2]:
# Drop columns that are redundant, identifier-only, or not useful for modeling
cols_to_drop = [
    'dep_AIRPRT_NM', 'arvl_AIRPRT_NM',
    'dep_CITY_METRO_IATA_CD', 'arvl_CITY_METRO_IATA_CD',
    'dep_CNTRY_CD', 'arvl_CNTRY_CD',
    'dep_WRLD_AREA_DOT_CD', 'arvl_WRLD_AREA_DOT_CD',
    'SCHD_LEG_ARVL_LCL_TMS',
    'ACTL_LEG_DEP_LCL_TMS',
    'SCHD_LEG_DEP_GMT_TMS',
    'SCHD_LEG_ARVL_GMT_TMS',
    'dep_hour',
    'arvl_hour',
]

df.drop(columns=cols_to_drop, inplace=True)
print(f"Columns remaining: {df.shape[1]}")
print(df.columns.tolist())

Columns remaining: 34
['OPERAT_AIRLN_IATA_CD', 'OPERAT_FLIGHT_NBR', 'SCHD_LEG_DEP_AIRPRT_IATA_CD', 'SCHD_LEG_ARVL_AIRPRT_IATA_CD', 'SCHD_LEG_DEP_LCL_TMS', 'LEG_DEP_VARNCE_MIN_QTY', 'LEG_ARVL_VARNCE_MIN_QTY', 'SCHD_FLEET_CD', 'ACTL_FLEET_CD', 'MINS_TO_SCHD_DEP_QTY', 'POST_NBR', 'DEP_STATUS_DESC', 'ARVL_STATUS_DESC', 'FLIFO_DELAY_REASON_CD', 'OP_PRE_STATUS_CD', 'OP_STATUS_CD', 'SUBSEQUENT_LEG_OF_DIVERT_IND', 'flight_duration_min', 'dep_LNGST_RUNWAY_FT_QTY', 'dep_ELEVATN_FT_QTY', 'dep_airport_latitude', 'dep_airport_longitude', 'arvl_LNGST_RUNWAY_FT_QTY', 'arvl_ELEVATN_FT_QTY', 'arvl_airport_latitude', 'arvl_airport_longitude', 'dep_wind_speed', 'dep_visibility', 'dep_precipitation', 'dep_wxcodes', 'arvl_wind_speed', 'arvl_visibility', 'arvl_precipitation', 'arvl_wxcodes']


In [3]:
# Convert departure timestamp to datetime
df['SCHD_LEG_DEP_LCL_TMS'] = pd.to_datetime(df['SCHD_LEG_DEP_LCL_TMS'])

# Extract departure date to identify unique flights per day
df['dep_date'] = df['SCHD_LEG_DEP_LCL_TMS'].dt.date

# Keep only pre-departure rows - updates posted before or at scheduled departure time
df_pre = df[df['MINS_TO_SCHD_DEP_QTY'] <= 0].copy()

print(f"Total rows          : {df.shape[0]:,}")
print(f"Pre-departure rows  : {df_pre.shape[0]:,}")

# For each unique flight per day keep the row with the highest POST_NBR
# This represents the last known state of the flight before departure
df = df_pre.sort_values('POST_NBR').groupby(
    ['OPERAT_FLIGHT_NBR', 'SCHD_LEG_DEP_AIRPRT_IATA_CD',
     'SCHD_LEG_ARVL_AIRPRT_IATA_CD', 'dep_date']
).last().reset_index()

# Drop the helper date column
df.drop(columns=['dep_date'], inplace=True)

print(f"After deduplication : {df.shape[0]:,} unique flights")

Total rows          : 2,062,951
Pre-departure rows  : 597,697
After deduplication : 225,656 unique flights


In [4]:
# Extract time-based features from scheduled departure timestamp
df['dep_hour'] = df['SCHD_LEG_DEP_LCL_TMS'].dt.hour
df['dep_dayofweek'] = df['SCHD_LEG_DEP_LCL_TMS'].dt.dayofweek
df['dep_month'] = df['SCHD_LEG_DEP_LCL_TMS'].dt.month

print(f"Shape: {df.shape}")
print(df[['SCHD_LEG_DEP_LCL_TMS', 'dep_hour', 'dep_dayofweek', 'dep_month']].head(3))

Shape: (225656, 37)
  SCHD_LEG_DEP_LCL_TMS  dep_hour  dep_dayofweek  dep_month
0  2024-06-10 10:05:00        10              0          6
1  2024-06-11 10:05:00        10              1          6
2  2024-06-14 10:05:00        10              4          6


In [5]:
# Create a binary flag indicating whether the delay reason is weather related
weather_codes = ['MTR', 'WXL', 'WXD', 'WXS', 'WXE', 'WXX',
                 'AWD', 'AWC', 'AWL', 'AWE', 'WEATHER']

def is_weather_delay(code):
    if pd.isna(code):
        return 0
    code = str(code).strip().upper()
    if code in weather_codes:
        return 1
    if 'WX' in code or 'WEATHER' in code or 'MTR' in code:
        return 1
    return 0

df['weather_delay_flag'] = df['FLIFO_DELAY_REASON_CD'].apply(is_weather_delay)

print(f"Weather delays: {df['weather_delay_flag'].sum():,} out of {len(df):,}")
print(df['weather_delay_flag'].value_counts())

Weather delays: 70,898 out of 225,656
weather_delay_flag
0    154758
1     70898
Name: count, dtype: int64


In [6]:
# Original change flag - any change of 1 minute or more
df['CHANGE_FLAG_1MIN'] = (df['LEG_ARVL_VARNCE_MIN_QTY'].abs() >= 1).astype(int)

# Operationally meaningful flag - flight arrives 10 or more minutes late
# This directly impacts Connect Assist hold decisions at DFW
# Minimum connection time at DFW is 40-45 minutes
# A 10 minute late arrival consumes 25% of that buffer
df['CHANGE_FLAG_10MIN'] = (df['LEG_ARVL_VARNCE_MIN_QTY'] >= 10).astype(int)

print("CHANGE_FLAG_1MIN distribution:")
print(df['CHANGE_FLAG_1MIN'].value_counts())
print(f"Change rate: {df['CHANGE_FLAG_1MIN'].mean()*100:.2f}%")

print("\nCHANGE_FLAG_10MIN distribution:")
print(df['CHANGE_FLAG_10MIN'].value_counts())
print(f"Significant late rate: {df['CHANGE_FLAG_10MIN'].mean()*100:.2f}%")

CHANGE_FLAG_1MIN distribution:
CHANGE_FLAG_1MIN
1    223384
0      2272
Name: count, dtype: int64
Change rate: 98.99%

CHANGE_FLAG_10MIN distribution:
CHANGE_FLAG_10MIN
1    181589
0     44067
Name: count, dtype: int64
Significant late rate: 80.47%


In [7]:
# Encode categorical columns to numeric for modeling
le = LabelEncoder()

categorical_cols = [
    'DEP_STATUS_DESC',
    'ARVL_STATUS_DESC',
    'OP_PRE_STATUS_CD',
    'OP_STATUS_CD',
    'OPERAT_AIRLN_IATA_CD',
    'SCHD_FLEET_CD',
    'ACTL_FLEET_CD',
    'SCHD_LEG_ARVL_AIRPRT_IATA_CD',
    'dep_wxcodes',
    'arvl_wxcodes',
]

for col in categorical_cols:
    df[col] = df[col].astype(str)
    df[col + '_ENC'] = le.fit_transform(df[col])

# Encode Y/N divert indicator as 1/0
df['SUBSEQUENT_LEG_OF_DIVERT_IND'] = df['SUBSEQUENT_LEG_OF_DIVERT_IND'].map({'Y': 1, 'N': 0})

print("Encoding complete.")
print(df[[c + '_ENC' for c in categorical_cols]].head(3))

Encoding complete.
   DEP_STATUS_DESC_ENC  ARVL_STATUS_DESC_ENC  OP_PRE_STATUS_CD_ENC  \
0                    1                     4                     3   
1                    1                     4                     3   
2                    1                     4                     3   

   OP_STATUS_CD_ENC  OPERAT_AIRLN_IATA_CD_ENC  SCHD_FLEET_CD_ENC  \
0                 3                         0                  0   
1                 3                         0                  0   
2                 3                         0                  0   

   ACTL_FLEET_CD_ENC  SCHD_LEG_ARVL_AIRPRT_IATA_CD_ENC  dep_wxcodes_ENC  \
0                  0                               194               44   
1                  0                               194               44   
2                  0                               194               44   

   arvl_wxcodes_ENC  
0                87  
1                94  
2                82  


In [9]:
df.drop(columns=['CHANGE_FLAG'], inplace=True, errors='ignore')

In [10]:
# Drop original categorical columns replaced by encoded versions
# Also drop identifier and timestamp columns not needed for modeling
cols_to_drop = [
    'DEP_STATUS_DESC', 'ARVL_STATUS_DESC', 'OP_PRE_STATUS_CD', 'OP_STATUS_CD',
    'OPERAT_AIRLN_IATA_CD', 'SCHD_FLEET_CD', 'ACTL_FLEET_CD',
    'SCHD_LEG_ARVL_AIRPRT_IATA_CD', 'dep_wxcodes', 'arvl_wxcodes',
    'FLIFO_DELAY_REASON_CD',
    'SCHD_LEG_DEP_LCL_TMS',
    'OPERAT_FLIGHT_NBR',
    'SCHD_LEG_DEP_AIRPRT_IATA_CD',
    'MINS_TO_SCHD_DEP_QTY',
]

df.drop(columns=cols_to_drop, inplace=True)
print(f"Shape after dropping: {df.shape}")
print(f"Columns: {df.columns.tolist()}")


Shape after dropping: (225656, 35)
Columns: ['LEG_DEP_VARNCE_MIN_QTY', 'LEG_ARVL_VARNCE_MIN_QTY', 'POST_NBR', 'SUBSEQUENT_LEG_OF_DIVERT_IND', 'flight_duration_min', 'dep_LNGST_RUNWAY_FT_QTY', 'dep_ELEVATN_FT_QTY', 'dep_airport_latitude', 'dep_airport_longitude', 'arvl_LNGST_RUNWAY_FT_QTY', 'arvl_ELEVATN_FT_QTY', 'arvl_airport_latitude', 'arvl_airport_longitude', 'dep_wind_speed', 'dep_visibility', 'dep_precipitation', 'arvl_wind_speed', 'arvl_visibility', 'arvl_precipitation', 'dep_hour', 'dep_dayofweek', 'dep_month', 'weather_delay_flag', 'CHANGE_FLAG_1MIN', 'CHANGE_FLAG_10MIN', 'DEP_STATUS_DESC_ENC', 'ARVL_STATUS_DESC_ENC', 'OP_PRE_STATUS_CD_ENC', 'OP_STATUS_CD_ENC', 'OPERAT_AIRLN_IATA_CD_ENC', 'SCHD_FLEET_CD_ENC', 'ACTL_FLEET_CD_ENC', 'SCHD_LEG_ARVL_AIRPRT_IATA_CD_ENC', 'dep_wxcodes_ENC', 'arvl_wxcodes_ENC']


In [11]:
# Fill missing departure weather with median since DFW data is mostly complete
df['dep_wind_speed'].fillna(df['dep_wind_speed'].median(), inplace=True)
df['dep_visibility'].fillna(df['dep_visibility'].median(), inplace=True)
df['dep_precipitation'].fillna(df['dep_precipitation'].median(), inplace=True)

# Drop rows with missing arrival weather - these are airports with no NOAA data
df.dropna(subset=['arvl_wind_speed', 'arvl_visibility', 'arvl_precipitation'], inplace=True)

# Fill missing divert indicator with 0
df['SUBSEQUENT_LEG_OF_DIVERT_IND'].fillna(0, inplace=True)

# Drop rows where target variable is missing
df.dropna(subset=['LEG_ARVL_VARNCE_MIN_QTY', 'LEG_DEP_VARNCE_MIN_QTY'], inplace=True)

df.reset_index(drop=True, inplace=True)

print(f"Shape after handling missing values: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Remaining missing values: {df.isnull().sum().sum()}")

Shape after handling missing values: 211,800 rows x 35 columns
Remaining missing values: 0


In [12]:
# Cap extreme outliers at 1st and 99th percentile to reduce model noise
cols_to_cap = ['LEG_DEP_VARNCE_MIN_QTY', 'LEG_ARVL_VARNCE_MIN_QTY']

for col in cols_to_cap:
    lower = df[col].quantile(0.01)
    upper = df[col].quantile(0.99)
    df[col] = df[col].clip(lower=lower, upper=upper)
    print(f"{col}: clipped to [{lower:.1f}, {upper:.1f}]")

LEG_DEP_VARNCE_MIN_QTY: clipped to [2.0, 488.0]
LEG_ARVL_VARNCE_MIN_QTY: clipped to [-14.0, 485.0]


In [13]:
# Save the final modeling dataset
output_path = os.path.join(DATA_FOLDER, "modeling_dataset.csv")
df.to_csv(output_path, index=False)

print(f"Saved: {output_path}")
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Columns: {df.columns.tolist()}")

Saved: /Users/shilppatel/Desktop/AA_FINAL_PROJECT/modeling_dataset.csv
Shape: 211,800 rows x 35 columns
Columns: ['LEG_DEP_VARNCE_MIN_QTY', 'LEG_ARVL_VARNCE_MIN_QTY', 'POST_NBR', 'SUBSEQUENT_LEG_OF_DIVERT_IND', 'flight_duration_min', 'dep_LNGST_RUNWAY_FT_QTY', 'dep_ELEVATN_FT_QTY', 'dep_airport_latitude', 'dep_airport_longitude', 'arvl_LNGST_RUNWAY_FT_QTY', 'arvl_ELEVATN_FT_QTY', 'arvl_airport_latitude', 'arvl_airport_longitude', 'dep_wind_speed', 'dep_visibility', 'dep_precipitation', 'arvl_wind_speed', 'arvl_visibility', 'arvl_precipitation', 'dep_hour', 'dep_dayofweek', 'dep_month', 'weather_delay_flag', 'CHANGE_FLAG_1MIN', 'CHANGE_FLAG_10MIN', 'DEP_STATUS_DESC_ENC', 'ARVL_STATUS_DESC_ENC', 'OP_PRE_STATUS_CD_ENC', 'OP_STATUS_CD_ENC', 'OPERAT_AIRLN_IATA_CD_ENC', 'SCHD_FLEET_CD_ENC', 'ACTL_FLEET_CD_ENC', 'SCHD_LEG_ARVL_AIRPRT_IATA_CD_ENC', 'dep_wxcodes_ENC', 'arvl_wxcodes_ENC']
